# Hands-on 35: Studi Kasus A - Smart Room Monitoring Lengkap

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 14.

Sistem lengkap: sensor/simulator -> MQTT -> subscriber Python (validasi, fitur
lag/rolling) -> klasifikasi nyaman/tidak_nyaman -> storage. Label mengikuti aturan buku:
nyaman jika suhu 24-28 C, kelembaban 45-65%, CO2 < 1000.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Latih Model (baseline LR vs Random Forest)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/case_a_train_room_model.py"], capture_output=True, text=True)
print(r.stdout)

[INFO] Distribusi label: {'tidak_nyaman': 1400, 'nyaman': 758}

=== Baseline Logistic Regression (accuracy): 0.6551 ===

=== Random Forest - Confusion Matrix ===
[[150   2]
 [  0 280]]

=== Random Forest - Classification Report ===
              precision    recall  f1-score   support

      nyaman       1.00      0.99      0.99       152
tidak_nyaman       0.99      1.00      1.00       280

    accuracy                           1.00       432
   macro avg       1.00      0.99      0.99       432
weighted avg       1.00      1.00      1.00       432

[SUKSES] Model disimpan ke: models/case_a_room_rf.joblib
[SUKSES] Laporan disimpan ke: outputs/reports/case_a_evaluation.csv



## 2. Deployment Mini (broker + subscriber + publisher)

In [3]:
influx_hapus("case_a_room")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = latar("scripts/case_a_room_subscriber.py", "case_a_sub_nb.log")
    time.sleep(2)
    pub = latar("scripts/case_a_room_publisher.py", "case_a_pub_nb.log")
    time.sleep(24)   # sekitar 12 pesan
    henti(pub, sub)
    print(log_teks("case_a_sub_nb.log"))
finally:
    broker.terminate()

Subscribe ke casea/room01/telemetry
[2026-10-22T13:28:55] -> status=tidak_nyaman
[2026-10-22T13:28:57] -> status=nyaman
[2026-10-22T13:28:59] -> status=nyaman
[2026-10-22T13:29:01] -> status=tidak_nyaman
[2026-10-22T13:29:03] -> status=nyaman
[2026-10-22T13:29:05] -> status=tidak_nyaman
[2026-10-22T13:29:07] -> status=nyaman
[2026-10-22T13:29:09] -> status=tidak_nyaman
[2026-10-22T13:29:11] -> status=nyaman
[2026-10-22T13:29:13] -> status=tidak_nyaman
[2026-10-22T13:29:15] -> status=tidak_nyaman
[2026-10-22T13:29:17] -> status=tidak_nyaman



## 3. Verifikasi Hasil

In [4]:
df = influx_df("case_a_room")
print(df.to_string(index=False))
print("\nDistribusi status:", df["pred_status"].value_counts().to_dict() if "pred_status" in df else "")

                    _time device_id  pred_status  co2_ppm  humidity_pct  temperature_c
2026-10-22 13:28:55+00:00    room01 tidak_nyaman   1145.0         63.35          31.47
2026-10-22 13:28:57+00:00    room01       nyaman    639.0         52.05          25.10
2026-10-22 13:28:59+00:00    room01       nyaman    671.0         61.51          26.51
2026-10-22 13:29:01+00:00    room01 tidak_nyaman   1165.0         47.86          30.79
2026-10-22 13:29:03+00:00    room01       nyaman    729.0         61.95          25.74
2026-10-22 13:29:05+00:00    room01 tidak_nyaman    623.0         50.71          31.18
2026-10-22 13:29:07+00:00    room01       nyaman    608.0         64.41          27.76
2026-10-22 13:29:09+00:00    room01 tidak_nyaman    894.0         62.55          31.74
2026-10-22 13:29:11+00:00    room01       nyaman   1212.0         70.90          24.98
2026-10-22 13:29:13+00:00    room01 tidak_nyaman    461.0         42.23          28.99
2026-10-22 13:29:15+00:00    room01 tidak_n

## Penutup

Studi kasus A menyatukan seluruh stack: Random Forest mencapai F1 sekitar 1.00 pada
label aturan kenyamanan, dan pipeline MQTT berjalan ujung ke ujung dengan storage CSV
(InfluxDB + Grafana opsional via `--influx`, panel: suhu, kelembaban, CO2, status,
histori status, alert). Model di `models/case_a_room_rf.joblib`.

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 37 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 6. Evaluasi

Metrik yang dipakai:

accuracy
precision
recall
F1
confusion matrix

Fokus utama:
- apakah tidak_nyaman sering lolos sebagai nyaman
- apakah model terlalu sensitif memberi tidak_nyaman

### 7. Dashboard real-time

Panel Grafana yang disarankan:

suhu real-time
kelembaban real-time
CO₂ real-time
status ruangan (nyaman/tidak_nyaman)
histori status
panel alert sederhana